# Final Holdout Evaluation

Previous experiments used a fixed 14-example validation benchmark to guide
dataset and architecture development.

Because that validation set influenced model development, it is no longer an
appropriate estimate of final generalization performance.

This notebook introduces a new holdout test set that has never been used for:

- training,
- dataset construction,
- architecture selection,
- encoder selection,
- threshold selection,
- or error-targeted augmentation.

The final test set is frozen before either selected model is evaluated.

Two configurations are tested:

1. **BGE Base** — highest validation quality
2. **MiniLM** — strongest latency/quality trade-off

No model changes will be made based on the results in this notebook.

In [1]:
import random

import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

from parallel_decider.dataset import (
    ROUTING_LABELS,
    load_routing_dataset,
)
from parallel_decider.projection import RoutingProjection
from parallel_decider.two_tower import TwoTowerDecisionHead

In [2]:
SEED = 42

device = "cuda" if torch.cuda.is_available() else "cpu"

def reset_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

reset_seed()

print("Device:", device)

Device: cuda


In [3]:
routing_hypotheses = {
    "needs_filesystem": "This task requires filesystem access.",
    "needs_git": "This task requires Git access.",
    "needs_shell": "This task requires shell execution.",
    "needs_browser": "This task requires browser access.",
    "needs_network": "This task requires network access.",
    "needs_database": "This task requires database access.",
    "needs_email": "This task requires email access.",
    "needs_calendar": "This task requires calendar access.",
}

## Final Test Set Integrity

The final holdout contains 40 previously unseen requests.

Before model evaluation, the dataset is checked for:

- the expected number of examples,
- exact overlap with training data,
- exact overlap with the development benchmark,
- and per-capability positive-label counts.

After these checks, the test set is considered frozen.

In [4]:
final_test_examples = load_routing_dataset(
    "../data/routing_final_test.jsonl"
)

training_examples = load_routing_dataset(
    "../data/routing_v2.jsonl"
)

validation_examples = load_routing_dataset(
    "../data/routing_benchmark.jsonl"
)

print("Final test examples:", len(final_test_examples))

assert len(final_test_examples) == 40

Final test examples: 40


In [5]:
training_states = {
    example.state
    for example in training_examples
}

validation_states = {
    example.state
    for example in validation_examples
}

final_states = {
    example.state
    for example in final_test_examples
}

training_overlap = final_states & training_states
validation_overlap = final_states & validation_states

print(
    "Training overlap:",
    len(training_overlap),
)

print(
    "Validation overlap:",
    len(validation_overlap),
)

assert not training_overlap
assert not validation_overlap

Training overlap: 0
Validation overlap: 0


In [6]:
for label in ROUTING_LABELS:
    positives = sum(
        example.labels[label]
        for example in final_test_examples
    )

    print(
        f"{label:<20} "
        f"positives={positives:>2} "
        f"negatives={len(final_test_examples) - positives:>2}"
    )

needs_filesystem     positives=16 negatives=24
needs_git            positives= 5 negatives=35
needs_shell          positives= 6 negatives=34
needs_browser        positives= 6 negatives=34
needs_network        positives=15 negatives=25
needs_database       positives= 6 negatives=34
needs_email          positives= 5 negatives=35
needs_calendar       positives= 4 negatives=36


## Final Evaluation Procedure

The final holdout has passed the integrity checks:

- 40 examples are present,
- no examples overlap with the training set,
- no examples overlap with the development benchmark,
- and all routing capabilities are represented.

The final evaluation compares the two selected encoder configurations:

1. **BGE Base EN v1.5**
2. **MiniLM L6 v2**

Both models are trained from scratch using the same 74-example training set,
random seed, routing projection, decision head, optimizer, learning rate,
number of epochs, and classification threshold.

The final test examples are used only after training is complete.

No model or dataset changes will be made based on the results of this holdout.

In [7]:
benchmark_states = {
    example.state
    for example in validation_examples
}

train_examples = [
    example
    for example in training_examples
    if example.state not in benchmark_states
]

print("Training examples:", len(train_examples))

assert len(train_examples) == 74

Training examples: 74


In [8]:
def compute_metrics(
    targets: torch.Tensor,
    predictions: torch.Tensor,
) -> pd.DataFrame:
    rows = []

    for index, label in enumerate(ROUTING_LABELS):
        y_true = targets[:, index]
        y_pred = predictions[:, index]

        tp = ((y_pred == 1) & (y_true == 1)).sum().item()
        fp = ((y_pred == 1) & (y_true == 0)).sum().item()
        fn = ((y_pred == 0) & (y_true == 1)).sum().item()
        tn = ((y_pred == 0) & (y_true == 0)).sum().item()

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )

        rows.append(
            {
                "label": label,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "tn": tn,
            }
        )

    return pd.DataFrame(rows)

In [9]:
def prepare_training_data(
    model_name: str,
    state_prefix: str = "",
    question_prefix: str = "",
):
    encoder = SentenceTransformer(
        model_name,
        device=device,
    )

    embedding_dim = encoder.get_embedding_dimension()

    train_states = [
        state_prefix + example.state
        for example in train_examples
    ]

    question_texts = [
        question_prefix + routing_hypotheses[label]
        for label in ROUTING_LABELS
    ]

    train_embeddings = encoder.encode(
        train_states,
        convert_to_tensor=True,
        show_progress_bar=True,
    ).detach().clone().float()

    question_embeddings = encoder.encode(
        question_texts,
        convert_to_tensor=True,
    ).detach().clone().float()

    train_targets = torch.tensor(
        [
            [
                example.labels[label]
                for label in ROUTING_LABELS
            ]
            for example in train_examples
        ],
        dtype=torch.float32,
        device=device,
    )

    return {
        "encoder": encoder,
        "embedding_dim": embedding_dim,
        "train_embeddings": train_embeddings,
        "question_embeddings": question_embeddings,
        "train_targets": train_targets,
        "state_prefix": state_prefix,
    }

In [10]:
def train_final_router(
    prepared_data,
    epochs: int = 100,
    learning_rate: float = 1e-3,
):
    reset_seed()

    projection = RoutingProjection(
        input_dim=prepared_data["embedding_dim"],
        output_dim=128,
    ).to(device)

    head = TwoTowerDecisionHead(
        embedding_dim=128,
        hidden_dim=128,
    ).to(device)

    optimizer = torch.optim.AdamW(
        list(projection.parameters())
        + list(head.parameters()),
        lr=learning_rate,
    )

    criterion = torch.nn.BCEWithLogitsLoss()

    train_embeddings = prepared_data["train_embeddings"]
    train_targets = prepared_data["train_targets"]
    question_embeddings = prepared_data["question_embeddings"]

    progress = tqdm(
        range(epochs),
        desc="Training final router",
    )

    for epoch in progress:
        epoch_loss = 0.0

        for state_embedding, targets in zip(
            train_embeddings,
            train_targets,
        ):
            optimizer.zero_grad()

            projected_state = projection(
                state_embedding
            )

            projected_questions = projection(
                question_embeddings
            )

            logits = head(
                state_embedding=projected_state,
                question_embeddings=projected_questions,
            )

            loss = criterion(
                logits,
                targets,
            )

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        average_loss = (
            epoch_loss / len(train_embeddings)
        )

        progress.set_postfix(
            loss=f"{average_loss:.4f}"
        )

    return projection, head

## Train Final Models

Both selected configurations are trained completely before the final holdout is
evaluated.

The sentence encoders remain frozen. Only the routing projection and decision
head are trained.

Trained routing weights are copied to CPU after each run so the GPU can be
cleared before training the next model.

In [11]:
minilm_prepared = prepare_training_data(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

minilm_projection, minilm_head = train_final_router(
    minilm_prepared
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Training final router:   0%|          | 0/100 [00:00<?, ?it/s]

In [13]:
minilm_checkpoint = {
    "model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_dim": minilm_prepared["embedding_dim"],
    "state_prefix": "",
    "projection": {
        key: value.detach().cpu().clone()
        for key, value in minilm_projection.state_dict().items()
    },
    "head": {
        key: value.detach().cpu().clone()
        for key, value in minilm_head.state_dict().items()
    },
}

In [14]:
import gc

del minilm_projection
del minilm_head
del minilm_prepared

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [15]:
bge_prepared = prepare_training_data(
    model_name="BAAI/bge-base-en-v1.5",
)

bge_projection, bge_head = train_final_router(
    bge_prepared
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Training final router:   0%|          | 0/100 [00:00<?, ?it/s]

In [16]:
bge_checkpoint = {
    "model_name": "BAAI/bge-base-en-v1.5",
    "embedding_dim": bge_prepared["embedding_dim"],
    "state_prefix": "",
    "projection": {
        key: value.detach().cpu().clone()
        for key, value in bge_projection.state_dict().items()
    },
    "head": {
        key: value.detach().cpu().clone()
        for key, value in bge_head.state_dict().items()
    },
}

In [17]:
del bge_projection
del bge_head
del bge_prepared

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [18]:
print(
    "MiniLM:",
    minilm_checkpoint["embedding_dim"],
)

print(
    "BGE Base:",
    bge_checkpoint["embedding_dim"],
)

MiniLM: 384
BGE Base: 768


## Final Holdout Evaluation

Both selected models have now been trained completely without using the final
40-example holdout set.

The next step evaluates the frozen MiniLM and BGE Base routing checkpoints on
the unseen holdout.

The evaluation procedure, threshold, routing hypotheses, and model architecture
are fixed before these results are observed.

The holdout will not be used for further tuning.

In [19]:
def evaluate_final_holdout(
    checkpoint,
    final_examples,
    threshold: float = 0.5,
    verbose: bool = True,
):
    encoder = SentenceTransformer(
        checkpoint["model_name"],
        device=device,
    )

    projection = RoutingProjection(
        input_dim=checkpoint["embedding_dim"],
        output_dim=128,
    ).to(device)

    head = TwoTowerDecisionHead(
        embedding_dim=128,
        hidden_dim=128,
    ).to(device)

    projection.load_state_dict(
        checkpoint["projection"]
    )

    head.load_state_dict(
        checkpoint["head"]
    )

    projection.eval()
    head.eval()

    state_prefix = checkpoint["state_prefix"]

    state_texts = [
        state_prefix + example.state
        for example in final_examples
    ]

    state_embeddings = encoder.encode(
        state_texts,
        convert_to_tensor=True,
        show_progress_bar=True,
    ).detach().clone().float()

    question_texts = [
        routing_hypotheses[label]
        for label in ROUTING_LABELS
    ]

    question_embeddings = encoder.encode(
        question_texts,
        convert_to_tensor=True,
        show_progress_bar=False,
    ).detach().clone().float()

    targets = torch.tensor(
        [
            [
                example.labels[label]
                for label in ROUTING_LABELS
            ]
            for example in final_examples
        ],
        dtype=torch.int64,
    )

    probability_rows = []

    with torch.no_grad():
        projected_questions = projection(
            question_embeddings
        )

        for index, state_embedding in enumerate(
            state_embeddings
        ):
            projected_state = projection(
                state_embedding
            )

            logits = head(
                state_embedding=projected_state,
                question_embeddings=projected_questions,
            )

            probabilities = torch.sigmoid(
                logits
            ).cpu()

            probability_rows.append(
                probabilities
            )

            if verbose:
                predictions = (
                    probabilities >= threshold
                ).int()

                print(f"\nTest {index + 1}")
                print(final_examples[index].state)

                for (
                    label,
                    probability,
                    prediction,
                    target,
                ) in zip(
                    ROUTING_LABELS,
                    probabilities,
                    predictions,
                    targets[index],
                ):
                    print(
                        f"{label:<20} "
                        f"p={probability.item():.3f} "
                        f"pred={int(prediction.item())} "
                        f"target={int(target.item())}"
                    )

    probabilities = torch.stack(
        probability_rows
    )

    predictions = (
        probabilities >= threshold
    ).int()

    accuracy = (
        predictions == targets
    ).float().mean().item()

    metrics = compute_metrics(
        targets,
        predictions,
    )

    result = {
        "accuracy": accuracy,
        "macro_precision": metrics["precision"].mean(),
        "macro_recall": metrics["recall"].mean(),
        "macro_f1": metrics["f1"].mean(),
        "metrics": metrics,
        "probabilities": probabilities,
        "predictions": predictions,
        "targets": targets,
    }

    del encoder
    del projection
    del head

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

In [20]:
minilm_final_result = evaluate_final_holdout(
    checkpoint=minilm_checkpoint,
    final_examples=final_test_examples,
    verbose=True,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Test 1
Explain the difference between SQLite WAL mode and rollback journals.
needs_filesystem     p=0.000 pred=0 target=0
needs_git            p=0.000 pred=0 target=0
needs_shell          p=0.000 pred=0 target=0
needs_browser        p=0.327 pred=0 target=0
needs_network        p=0.000 pred=0 target=0
needs_database       p=0.000 pred=0 target=0
needs_email          p=0.000 pred=0 target=0
needs_calendar       p=0.000 pred=0 target=0

Test 2
Look up invoice 78342 in the billing database and report whether it has been paid.
needs_filesystem     p=0.855 pred=1 target=0
needs_git            p=0.000 pred=0 target=0
needs_shell          p=0.000 pred=0 target=0
needs_browser        p=0.000 pred=0 target=0
needs_network        p=0.822 pred=1 target=0
needs_database       p=1.000 pred=1 target=1
needs_email          p=0.000 pred=0 target=0
needs_calendar       p=0.000 pred=0 target=0

Test 3
Query the inventory database for products below the reorder threshold and save the result as a local JS

In [21]:
print(
    f"MiniLM accuracy:        "
    f"{minilm_final_result['accuracy']:.3%}"
)

print(
    f"MiniLM macro precision: "
    f"{minilm_final_result['macro_precision']:.3f}"
)

print(
    f"MiniLM macro recall:    "
    f"{minilm_final_result['macro_recall']:.3f}"
)

print(
    f"MiniLM macro F1:        "
    f"{minilm_final_result['macro_f1']:.3f}"
)

MiniLM accuracy:        92.500%
MiniLM macro precision: 0.854
MiniLM macro recall:    0.822
MiniLM macro F1:        0.819


In [22]:
bge_final_result = evaluate_final_holdout(
    checkpoint=bge_checkpoint,
    final_examples=final_test_examples,
    verbose=True,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Test 1
Explain the difference between SQLite WAL mode and rollback journals.
needs_filesystem     p=0.000 pred=0 target=0
needs_git            p=0.000 pred=0 target=0
needs_shell          p=0.000 pred=0 target=0
needs_browser        p=0.000 pred=0 target=0
needs_network        p=0.000 pred=0 target=0
needs_database       p=0.000 pred=0 target=0
needs_email          p=0.000 pred=0 target=0
needs_calendar       p=0.000 pred=0 target=0

Test 2
Look up invoice 78342 in the billing database and report whether it has been paid.
needs_filesystem     p=0.011 pred=0 target=0
needs_git            p=0.000 pred=0 target=0
needs_shell          p=0.000 pred=0 target=0
needs_browser        p=0.000 pred=0 target=0
needs_network        p=0.000 pred=0 target=0
needs_database       p=1.000 pred=1 target=1
needs_email          p=0.000 pred=0 target=0
needs_calendar       p=0.000 pred=0 target=0

Test 3
Query the inventory database for products below the reorder threshold and save the result as a local JS

In [23]:
print(
    f"BGE Base accuracy:        "
    f"{bge_final_result['accuracy']:.3%}"
)

print(
    f"BGE Base macro precision: "
    f"{bge_final_result['macro_precision']:.3f}"
)

print(
    f"BGE Base macro recall:    "
    f"{bge_final_result['macro_recall']:.3f}"
)

print(
    f"BGE Base macro F1:        "
    f"{bge_final_result['macro_f1']:.3f}"
)

BGE Base accuracy:        96.875%
BGE Base macro precision: 0.964
BGE Base macro recall:    0.884
BGE Base macro F1:        0.914


In [24]:
minilm_final_result["metrics"][
    [
        "label",
        "precision",
        "recall",
        "f1",
        "tp",
        "fp",
        "fn",
    ]
].round(3)

,label,precision,recall,f1,tp,fp,fn
0,needs_filesystem,0.737,0.875,0.800,14,5,2
1,needs_git,1.000,1.000,1.000,5,0,0
2,needs_shell,0.625,0.833,0.714,5,3,1
3,needs_browser,0.833,0.833,0.833,5,1,1
4,needs_network,0.800,0.800,0.800,12,3,3
5,needs_database,0.833,0.833,0.833,5,1,1
6,needs_email,1.000,0.400,0.571,2,0,3
7,needs_calendar,1.000,1.000,1.000,4,0,0


In [25]:
bge_final_result["metrics"][
    [
        "label",
        "precision",
        "recall",
        "f1",
        "tp",
        "fp",
        "fn",
    ]
].round(3)

,label,precision,recall,f1,tp,fp,fn
0,needs_filesystem,0.882,0.938,0.909,15,2,1
1,needs_git,0.833,1.000,0.909,5,1,0
2,needs_shell,1.000,0.833,0.909,5,0,1
3,needs_browser,1.000,1.000,1.000,6,0,0
4,needs_network,1.000,0.867,0.929,13,0,2
5,needs_database,1.000,0.833,0.909,5,0,1
6,needs_email,1.000,0.600,0.750,3,0,2
7,needs_calendar,1.000,1.000,1.000,4,0,0


## Final Holdout Results

The two selected routing configurations were evaluated on a frozen set of 40
previously unseen requests, representing 320 independent capability decisions.

The holdout set was not used for training, architecture selection, encoder
selection, threshold selection, or targeted dataset augmentation.

| Model | Accuracy | Macro Precision | Macro Recall | Macro F1 | Errors |
| --- | ---: | ---: | ---: | ---: | ---: |
| MiniLM L6 v2 | 92.50% | 0.854 | 0.822 | 0.819 | 24 / 320 |
| BGE Base EN v1.5 | **96.88%** | **0.964** | **0.884** | **0.914** | **10 / 320** |

BGE Base reduces the number of incorrect routing decisions from 24 to 10,
corresponding to approximately 58% fewer errors than MiniLM on the final
holdout.

### Per-Capability Performance — BGE Base

| Capability | Precision | Recall | F1 |
| --- | ---: | ---: | ---: |
| Filesystem | 0.882 | 0.938 | 0.909 |
| Git | 0.833 | 1.000 | 0.909 |
| Shell | 1.000 | 0.833 | 0.909 |
| Browser | 1.000 | 1.000 | 1.000 |
| Network | 1.000 | 0.867 | 0.929 |
| Database | 1.000 | 0.833 | 0.909 |
| Email | 1.000 | 0.600 | 0.750 |
| Calendar | 1.000 | 1.000 | 1.000 |

BGE Base achieves perfect precision for six of the eight capabilities and
perfect recall for Git, browser, and calendar.

The largest remaining weakness is email routing. Email precision is perfect,
but recall is only 0.60, meaning that the model misses some requests that
require email access.

### Validation vs Final Holdout

Performance decreases on the larger unseen holdout, as expected.

| Model | Validation Accuracy | Final Accuracy |
| --- | ---: | ---: |
| MiniLM | 99.11% | 92.50% |
| BGE Base | 100.00% | 96.88% |

The smaller drop for BGE Base suggests that its learned representation
generalizes better to unseen routing requests.

The final holdout therefore supports the encoder-selection result from the
development experiments: BGE Base provides the strongest overall routing
quality, while MiniLM remains the lower-latency alternative.

These results should still be interpreted in the context of a relatively small,
hand-designed dataset. They demonstrate the viability of the architecture, not
a claim of production-level routing reliability.